# Task 3: RAG with Unsloth Dynamic 4-bit Quantization

**Arch Technologies — Generative AI Internship (Month 2)**

This Colab notebook builds a memory-efficient **Retrieval-Augmented Generation (RAG)** pipeline:

1. Load a **dynamic 4-bit** Unsloth model (`*-unsloth-bnb-4bit`)
2. Index domain-specific documents with **sentence-transformers + FAISS**
3. Retrieve relevant chunks for a user query
4. Generate **grounded** answers with the quantized LLM
5. Track **VRAM** so the pipeline stays within a free Colab T4 (~15 GB)

### How to run
1. Runtime → Change runtime type → **T4 GPU**
2. Runtime → **Run all**
3. Ask questions in the demo cell or the Gradio UI at the end

References: [Unsloth Dynamic 4-bit](https://unsloth.ai/blog/dynamic-4bit) · [Unsloth 4-bit collection](https://huggingface.co/collections/unsloth/unsloth-4-bit-dynamic-quants)

## 1. GPU check

Unsloth 4-bit inference needs a CUDA GPU. If this cell fails, enable a T4 GPU and re-run.

In [ ]:

import torch

assert torch.cuda.is_available(), (
    "No GPU detected. In Colab: Runtime → Change runtime type → T4 GPU, then Runtime → Run all."
)
props = torch.cuda.get_device_properties(0)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {props.total_memory / 1024**3:.2f} GB")
print(f"CUDA: {torch.version.cuda} | PyTorch: {torch.__version__}")

GPU: Tesla T4
VRAM: 14.56 GB
CUDA: 12.8 | PyTorch: 2.11.0+cu128


## 2. Install Unsloth + RAG libraries

The first install cell follows Unsloth's official Colab recipe. Extra packages cover chunking, embeddings, FAISS, PDF ingest, and a lightweight UI.

In [ ]:
%%capture
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install -q unsloth
else:
    import torch
    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + {"2.10": "0.0.34", "2.9": "0.0.33.post1", "2.8": "0.0.32.post2"}.get(v, "0.0.34")
    !pip install -q sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install -q --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install -q --no-deps --upgrade "torchao>=0.16.0"
    !pip install -q transformers==4.56.2
    !pip install -q --no-deps trl==0.22.2

!pip install -q sentence-transformers faiss-cpu langchain-text-splitters pypdf gradio

## 3. Why Unsloth dynamic 4-bit?

Standard BitsAndBytes 4-bit (`bnb-4bit`) quantizes **all** linear weights to NF4. That saves a lot of VRAM, but some layers are quantization-sensitive (attention output projections, early layers, norms). Naive 4-bit can drop accuracy.

**Unsloth Dynamic 4-bit** still uses BitsAndBytes NF4, but **selectively skips quantization** for those critical parameters and keeps them at higher precision. The result:

| Variant | Typical naming | Accuracy | VRAM vs standard 4-bit |
|---|---|---|---|
| Full precision | no suffix | highest | ~4× more |
| Standard BnB 4-bit | `*-bnb-4bit` | lower | baseline |
| **Unsloth dynamic 4-bit** | **`*-unsloth-bnb-4bit`** | close to 16-bit | **<10% more than BnB 4-bit** |

This notebook loads `unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit` so it fits comfortably on a free T4 **and** uses the dynamic quant the task asks for. Swap to `unsloth/Llama-3.1-8B-Instruct-unsloth-bnb-4bit` if you have extra VRAM.

## 4. Load the dynamic 4-bit Unsloth model

In [ ]:
from unsloth import FastLanguageModel
import torch
import gc

MAX_SEQ_LENGTH = 2048
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit"

def vram_report(tag: str) -> None:
    if not torch.cuda.is_available():
        print(f"[{tag}] CUDA not available")
        return
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(
        f"[{tag}] allocated={allocated:.2f} GB | reserved={reserved:.2f} GB | "
        f"device total={total:.2f} GB | free≈{total - reserved:.2f} GB"
    )

torch.cuda.empty_cache()
gc.collect()
vram_report("before model load")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # fp16 on T4, bf16 on Ampere+
    load_in_4bit=True,   # BitsAndBytes NF4 + Unsloth dynamic skip list
)
FastLanguageModel.for_inference(model)

print("Loaded:", MODEL_NAME)
print("Quantization:", getattr(model.config, "quantization_config", None).__class__.__name__)
vram_report("after Unsloth 4-bit load")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[before model load] allocated=0.01 GB | reserved=0.02 GB | device total=14.56 GB | free≈14.54 GB
==((====))==  Unsloth 2026.8.19: Fast Llama patching. Transformers: 5.15.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit as a legacy tokenizer.


Loaded: unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit
Quantization: dict
[after Unsloth 4-bit load] allocated=2.24 GB | reserved=2.27 GB | device total=14.56 GB | free≈12.29 GB


Inspect which modules stayed at higher precision. Dynamic quants keep a subset of parameters (often output projections / sensitive layers) out of 4-bit.

In [ ]:
from collections import Counter

dtype_counts = Counter()
fourbit_linear = 0
other_modules = 0

for name, module in model.named_modules():
    cls = module.__class__.__name__
    if "Linear4bit" in cls or "Params4bit" in cls:
        fourbit_linear += 1
        continue
    if hasattr(module, "weight") and getattr(module.weight, "dtype", None) is not None:
        dtype_counts[str(module.weight.dtype)] += 1
        other_modules += 1

print("4-bit Linear modules:", fourbit_linear)
print("Modules with a dense weight tensor:", other_modules)
print("Dense weight dtypes:", dict(dtype_counts))
print()
print("This mix is expected: dynamic 4-bit compresses most linears to NF4")
print("and preserves higher precision on sensitive parameters.")

4-bit Linear modules: 193
Modules with a dense weight tensor: 62
Dense weight dtypes: {'torch.float16': 62}

This mix is expected: dynamic 4-bit compresses most linears to NF4
and preserves higher precision on sensitive parameters.


## 5. Domain knowledge base

The corpus is a small **LLM systems / intern lab** knowledge base with unique facts (GPU quotas, model IDs, lab rules). Those facts are **not** in Llama's pretraining in this exact form, so a correct answer is evidence that retrieval is working.

You can also upload extra `.txt` / `.pdf` files later.

In [ ]:
from pathlib import Path

KB_DIR = Path("knowledge_base")
KB_DIR.mkdir(exist_ok=True)

DOCUMENTS = {
    "unsloth_dynamic_4bit.md": '''
# Unsloth Dynamic 4-bit Quantization (ArchTech Lab Notes)

Unsloth Dynamic 4-bit builds on BitsAndBytes NF4. Instead of quantizing every linear layer,
it skips quantization for parameters that produce large reconstruction error.

Key lab facts:
- Dynamic 4-bit models on Hugging Face use the suffix unsloth-bnb-4bit.
- Standard BitsAndBytes-only models use the suffix bnb-4bit without the unsloth infix.
- Dynamic 4-bit typically uses less than 10% more VRAM than standard BnB 4-bit.
- For Llama-family models, Unsloth analysis found that attention output projections
  (o_proj) on most layers should remain at higher precision.
- ArchTech intern default model for Task 3 is Llama-3.2-3B-Instruct-unsloth-bnb-4bit.
- On a 15 GB T4, keep max_seq_length at 2048 and generate at most 256 new tokens per answer.
''',
    "rag_pipeline.md": '''
# Retrieval-Augmented Generation at ArchTech

RAG reduces hallucinations by injecting retrieved evidence into the prompt.

ArchTech RAG contract (Month 2):
- Chunk size: 450 characters with 80-character overlap.
- Embedding model: sentence-transformers/all-MiniLM-L6-v2 (384 dimensions).
- Vector index: FAISS IndexFlatIP on L2-normalized vectors (cosine similarity).
- Retrieve top-k = 4 chunks unless the query is a yes/no fact check, then k = 2.
- Generator must refuse if no chunk scores above similarity 0.25.
- Answers must cite source filenames from the retrieved metadata.
- Never fine-tune the generator for this task; retrieval is the knowledge update path.
''',
    "archtech_intern_handbook.md": '''
# Arch Technologies Intern GPU Handbook

These rules are unique to the internship and must be retrieved, not guessed.

- Intern GPU code name: Nimbus-T4.
- Daily Colab GPU budget per intern: 6 hours.
- Knowledge-base files live in /workspace/knowledge_base during evaluation.
- The official retrieval temperature for grounded answers is 0.2.
- The unofficial "ungrounded" ablation temperature is 0.7 (used only for comparison).
- Lab ticket prefix for RAG bugs is AT-RAG-2026.
- If VRAM reserved exceeds 12 GB on a T4, interns must drop to Llama-3.2-1B-Instruct-unsloth-bnb-4bit.
- Domain owner for this knowledge base is Mentor Desk 4 (Generative AI track).
''',
    "faiss_ops.md": '''
# FAISS Operations Cheatsheet

FAISS stores embedding vectors and returns nearest neighbors by inner product or L2.

ArchTech notes:
- IndexFlatIP is exact search and is preferred for corpora under 50,000 chunks.
- Normalize embeddings to unit length before adding them so inner product equals cosine.
- Persist the index with faiss.write_index(index, "archtech_rag.faiss").
- Store chunk text and metadata in a parallel JSONL file named chunks.jsonl.
- Re-embed the whole corpus if the embedding model changes; do not mix vector spaces.
- MiniLM embeddings for this lab are 384-D; a mismatch with index dimension is a fatal error.
''',
}

for name, text in DOCUMENTS.items():
    (KB_DIR / name).write_text(text.strip() + "\n", encoding="utf-8")

print(f"Wrote {len(DOCUMENTS)} documents to {KB_DIR.resolve()}")
for p in sorted(KB_DIR.glob("*")):
    print(f"  - {p.name}: {p.stat().st_size} bytes")

Wrote 4 documents to /content/knowledge_base
  - archtech_intern_handbook.md: 659 bytes
  - faiss_ops.md: 632 bytes
  - rag_pipeline.md: 668 bytes
  - unsloth_dynamic_4bit.md: 808 bytes


## 6. Chunk documents

`RecursiveCharacterTextSplitter` keeps paragraphs together when possible, then falls back to sentences and words. Overlap preserves sentences that straddle chunk boundaries.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=80,
    separators=["\n## ", "\n# ", "\n\n", "\n", ". ", " ", ""],
)

chunks = []
for path in sorted(KB_DIR.glob("*")):
    if path.suffix.lower() not in {".md", ".txt", ".pdf"}:
        continue
    if path.suffix.lower() == ".pdf":
        from pypdf import PdfReader
        text = "\n".join(page.extract_text() or "" for page in PdfReader(str(path)).pages)
    else:
        text = path.read_text(encoding="utf-8")
    for i, piece in enumerate(splitter.split_text(text)):
        chunks.append(
            {
                "text": piece.strip(),
                "source": path.name,
                "chunk_id": f"{path.stem}-{i}",
            }
        )

print(f"Indexed files: {len(list(KB_DIR.glob('*')))}")
print(f"Chunks: {len(chunks)}")
print("Sample chunk:")
print(chunks[0]["chunk_id"], "←", chunks[0]["source"])
print(chunks[0]["text"][:400])

Indexed files: 4
Chunks: 12
Sample chunk:
archtech_intern_handbook-0 ← archtech_intern_handbook.md
# Arch Technologies Intern GPU Handbook

These rules are unique to the internship and must be retrieved, not guessed.


## 7. Embed + FAISS index

MiniLM runs on CPU so the GPU stays free for the LLM. Vectors are normalized; FAISS `IndexFlatIP` then ranks by cosine similarity.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")

texts = [c["text"] for c in chunks]
embeddings = embedder.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype("float32")

assert embeddings.shape[1] == 384, embeddings.shape

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
faiss.write_index(index, "archtech_rag.faiss")

print(f"FAISS vectors: {index.ntotal} x {index.d}")
print(f"Embedding model: {EMBED_MODEL_NAME}")
vram_report("after FAISS index (embeddings on CPU)")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS vectors: 12 x 384
Embedding model: sentence-transformers/all-MiniLM-L6-v2
[after FAISS index (embeddings on CPU)] allocated=2.24 GB | reserved=2.27 GB | device total=14.56 GB | free≈12.29 GB


## 8. Retrieval + grounded generation

In [ ]:
from typing import Any

SIMILARITY_FLOOR = 0.25
TOP_K = 4

SYSTEM_PROMPT = (
    "You are ArchTech's RAG assistant. Answer using ONLY the retrieved context. "
    "If the context is missing the answer, say you do not have that information. "
    "Cite source filenames. Do not invent lab policies, model names, or numeric limits."
)


def retrieve(query: str, k: int = TOP_K) -> list[dict[str, Any]]:
    q = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(q, k)
    hits = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx < 0:
            continue
        item = dict(chunks[int(idx)])
        item["score"] = float(score)
        hits.append(item)
    return hits


def format_context(hits: list[dict[str, Any]]) -> str:
    blocks = []
    for h in hits:
        blocks.append(f"[source={h['source']} | id={h['chunk_id']} | score={h['score']:.3f}]\n{h['text']}")
    return "\n\n".join(blocks)


def generate_from_prompt(user_prompt: str, max_new_tokens: int = 256, temperature: float = 0.2) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            top_p=0.9,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0, inputs["input_ids"].shape[-1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def rag_answer(query: str, k: int = TOP_K, max_new_tokens: int = 256) -> dict[str, Any]:
    hits = retrieve(query, k=k)
    usable = [h for h in hits if h["score"] >= SIMILARITY_FLOOR]
    if not usable:
        return {
            "answer": "I do not have that information in the knowledge base (no chunk passed the similarity floor).",
            "hits": hits,
            "used_rag": True,
        }
    context = format_context(usable)
    user_prompt = (
        f"Retrieved context:\n{context}\n\n"
        f"Question: {query}\n\n"
        "Write a concise grounded answer."
    )
    answer = generate_from_prompt(user_prompt, max_new_tokens=max_new_tokens, temperature=0.2)
    return {"answer": answer, "hits": usable, "used_rag": True}


print("Retriever and generator ready.")

Retriever and generator ready.


## 9. Demo queries

These questions are answered by the knowledge base. The last one is intentionally out of domain so the model should refuse.

In [ ]:
from pprint import pprint

DEMO_QUESTIONS = [
    "What suffix do Unsloth dynamic 4-bit models use on Hugging Face?",
    "What is the ArchTech intern GPU code name and daily Colab budget?",
    "Which embedding model and FAISS index should we use, and what is top-k?",
    "What is the RAG bug ticket prefix?",
    "Who won the 2018 FIFA World Cup, according to the knowledge base?",
]

for q in DEMO_QUESTIONS:
    print("=" * 80)
    print("Q:", q)
    result = rag_answer(q)
    print("\nRetrieved:")
    for h in result["hits"]:
        print(f"  {h['score']:.3f}  {h['source']}  {h['chunk_id']}")
    print("\nA:", result["answer"])
    print()

Q: What suffix do Unsloth dynamic 4-bit models use on Hugging Face?


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Retrieved:
  0.609  unsloth_dynamic_4bit.md  unsloth_dynamic_4bit-1
  0.423  unsloth_dynamic_4bit.md  unsloth_dynamic_4bit-0

A: The suffix used by Unsloth dynamic 4-bit models on Hugging Face is "unsloth-bnb-4bit".

Q: What is the ArchTech intern GPU code name and daily Colab budget?


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Retrieved:
  0.635  archtech_intern_handbook.md  archtech_intern_handbook-0
  0.611  archtech_intern_handbook.md  archtech_intern_handbook-1
  0.311  archtech_intern_handbook.md  archtech_intern_handbook-2
  0.272  unsloth_dynamic_4bit.md  unsloth_dynamic_4bit-2

A: The ArchTech intern GPU code name is Nimbus-T4, and the daily Colab GPU budget per intern is 6 hours.

Q: Which embedding model and FAISS index should we use, and what is top-k?


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Retrieved:
  0.509  faiss_ops.md  faiss_ops-0
  0.431  rag_pipeline.md  rag_pipeline-1
  0.385  faiss_ops.md  faiss_ops-2
  0.383  faiss_ops.md  faiss_ops-1

A: Based on the provided context, the recommended embedding model is the sentence-transformers/all-MiniLM-L6-v2 (384 dimensions), and the FAISS index is IndexFlatIP on L2-normalized vectors (cosine similarity). The top-k is 4, unless the query is a yes/no fact check, in which case k = 2.

Q: What is the RAG bug ticket prefix?

Retrieved:
  0.568  archtech_intern_handbook.md  archtech_intern_handbook-2
  0.253  archtech_intern_handbook.md  archtech_intern_handbook-1

A: The RAG bug ticket prefix is AT-RAG-2026. [archtech_intern_handbook-2]

Q: Who won the 2018 FIFA World Cup, according to the knowledge base?

Retrieved:
  0.240  rag_pipeline.md  rag_pipeline-2
  0.137  archtech_intern_handbook.md  archtech_intern_handbook-1
  0.124  rag_pipeline.md  rag_pipeline-1
  0.114  archtech_intern_handbook.md  archtech_intern_handbook-0

A

## 10. Ablation: RAG vs ungrounded LLM

Same questions, no retrieved context. The intern-handbook facts should fail without RAG.

In [ ]:
ABLATION = [
    "What is the ArchTech intern GPU code name?",
    "What is the official retrieval temperature for grounded answers?",
]

for q in ABLATION:
    print("=" * 80)
    print("Q:", q)
    ungrounded = generate_from_prompt(
        f"Question: {q}\nAnswer using your own parameters. If unsure, guess.",
        temperature=0.7,
    )
    grounded = rag_answer(q)
    print("\nUNGBOUNDED:\n", ungrounded)
    print("\nRAG:\n", grounded["answer"])
    print()

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the ArchTech intern GPU code name?


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



UNGBOUNDED:
 I don't have information on ArchTech intern GPU code names.

RAG:
 The ArchTech intern GPU code name is Nimbus-T4. [archtech_intern_handbook-1]

Q: What is the official retrieval temperature for grounded answers?


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



UNGBOUNDED:
 I don't have information on the official retrieval temperature for grounded answers.

RAG:
 The official retrieval temperature for grounded answers is 0.2. [archtech_intern_handbook-1]



## 11. Optional: add your own PDF or text file

Upload a file, rebuild the index, then ask questions about that document.

In [ ]:
from pathlib import Path

# Set True only when you want the Colab file picker (do not leave True during Runtime → Run all).
ADD_EXTRA_FILES = False
uploaded = {}

if ADD_EXTRA_FILES:
    from google.colab import files
    print("Choose a .pdf / .txt / .md to add to the knowledge base.")
    uploaded = files.upload()
else:
    print("Skipping upload. Set ADD_EXTRA_FILES = True and re-run this cell to add documents.")

added = 0
for fname, raw in uploaded.items():
    dest = KB_DIR / Path(fname).name
    dest.write_bytes(raw)
    added += 1
    print("Saved", dest)

if added:
    # Re-run chunking + indexing by executing the earlier cells' logic
    chunks.clear()
    for path in sorted(KB_DIR.glob("*")):
        if path.suffix.lower() not in {".md", ".txt", ".pdf"}:
            continue
        if path.suffix.lower() == ".pdf":
            from pypdf import PdfReader
            text = "\n".join(page.extract_text() or "" for page in PdfReader(str(path)).pages)
        else:
            text = path.read_text(encoding="utf-8")
        for i, piece in enumerate(splitter.split_text(text)):
            chunks.append({"text": piece.strip(), "source": path.name, "chunk_id": f"{path.stem}-{i}"})
    texts = [c["text"] for c in chunks]
    embeddings = embedder.encode(texts, convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    index.reset()
    index.add(embeddings)
    print(f"Rebuilt index with {index.ntotal} chunks.")
else:
    print("No extra files added.")

Skipping upload. Set ADD_EXTRA_FILES = True and re-run this cell to add documents.
No extra files added.


## 12. Interactive Gradio UI

In [ ]:
import gradio as gr

def ui_ask(question, k):
    if not question or not question.strip():
        return "Enter a question.", ""
    result = rag_answer(question.strip(), k=int(k))
    sources = "\n".join(
        f"- {h['source']} ({h['chunk_id']}, score={h['score']:.3f})\n  {h['text'][:240]}..."
        for h in result["hits"]
    )
    return result["answer"], sources

demo = gr.Interface(
    fn=ui_ask,
    inputs=[
        gr.Textbox(label="Question", lines=3, placeholder="e.g. What is the intern GPU code name?"),
        gr.Slider(1, 6, value=4, step=1, label="top-k chunks"),
    ],
    outputs=[
        gr.Textbox(label="Grounded answer", lines=8),
        gr.Textbox(label="Retrieved chunks", lines=12),
    ],
    title="ArchTech RAG — Unsloth Dynamic 4-bit",
    description=f"Generator: {MODEL_NAME} · Embeddings: {EMBED_MODEL_NAME} · Index: FAISS IndexFlatIP",
)

vram_report("before Gradio")
demo.launch(share=True, debug=False)

[before Gradio] allocated=2.36 GB | reserved=2.62 GB | device total=14.56 GB | free≈11.95 GB
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f97128a945a3ceb3fa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 13. Submission checklist

- [x] Colab notebook with a **dynamic 4-bit** Unsloth model (`unsloth-bnb-4bit`)
- [x] Domain documents indexed with MiniLM + FAISS
- [x] Retrieval of relevant chunks for a query
- [x] Quantized LLM generates **grounded** answers (and refuses out-of-corpus questions)
- [x] VRAM reporting around model load and indexing
- [x] Optional PDF/text ingest + Gradio demo

**Suggested GitHub layout:** `month-2/task-3-rag-unsloth/` with this notebook and a short README.

When you screenshot results for LinkedIn/GitHub, include: GPU name, VRAM after load, a handbook question, and the World Cup refusal.